In [1]:
import os

# Fix TF 2.18 + protobuf runtime crash in this Kaggle environment:
# Ensure pure-Python protobuf is used and modern protobuf codepath is disabled BEFORE any TF/protobuf import.
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION_VERSION"] = "2"
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION_DISABLE_CXX"] = "1"
os.environ["TF_USE_LEGACY_PROTOBUF"] = "1"
os.environ["TF_USE_MODERN_PROTOBUF"] = "0"
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("TF_XLA_FLAGS", "--tf_xla_auto_jit=0")
os.environ.setdefault("TF_ENABLE_ONEDNN_OPTS", "0")

import sys

assert (
    "tensorflow" not in sys.modules
), "TensorFlow was imported before protobuf env vars were set."

# Lock protobuf runtime early (prevents TF from pulling in C++/upb protobuf paths).
import google.protobuf  # noqa: F401
from google.protobuf import message as _pb_message  # noqa: F401
from google.protobuf import descriptor_pool as _pb_descriptor_pool  # noqa: F401

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from random import sample
import cv2
import random

random.seed(42)
np.random.seed(42)

CANDIDATES = [
    "../input/petfinder-pawpularity-score",
    "/kaggle/input/petfinder-pawpularity-score",
    "/kaggle/data/petfinder-pawpularity-score",
    "/kaggle/data/petfinder-pawpularity-score/petfinder-pawpularity-score",
    "/kaggle/input/petfinder-pawpularity-score/petfinder-pawpularity-score",
]
BASE = None
for c in CANDIDATES:
    if os.path.exists(os.path.join(c, "train.csv")):
        BASE = c
        break
if BASE is None:
    raise FileNotFoundError(
        "Could not locate train.csv under expected Kaggle dataset paths."
    )

TRAIN_IMG_DIR = os.path.join(BASE, "train")
TEST_IMG_DIR = os.path.join(BASE, "test")
if not os.path.isdir(TRAIN_IMG_DIR):
    raise FileNotFoundError(f"Train image dir not found: {TRAIN_IMG_DIR}")
if not os.path.isdir(TEST_IMG_DIR):
    raise FileNotFoundError(f"Test image dir not found: {TEST_IMG_DIR}")



In [2]:
dat = pd.read_csv(os.path.join(BASE, "train.csv"))



In [3]:
y = dat["Pawpularity"].to_numpy()
y = y / 100.0




In [4]:
def read_image_rgb_128(path):
    img = cv2.imread(path)
    if img is None:
        return None
    img = cv2.resize(img, (128, 128), interpolation=cv2.INTER_AREA)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img.astype(np.float32) / 255.0




In [5]:
class data:
    def __init__(self, path, ids, x=128, y=128, labels=None):
        self.x = x
        self.y = y
        self.labels = labels
        self.image_list = list(ids)
        self.path = path
        self.batch = 0

    def load_batch(self, batch_size=1, shuffle=False):
        if shuffle:
            batch_list = sample(self.image_list, batch_size)
        else:
            b = self.batch
            batch_list = self.image_list[b * batch_size : (b + 1) * batch_size]
            self.batch = b + 1
            if self.batch > len(self.image_list) // batch_size:
                self.batch = 0

        images = []
        good_ids = []
        for image_id in batch_list:
            img_path = os.path.join(self.path, image_id + ".jpg")
            img = read_image_rgb_128(img_path)
            if img is None:
                continue
            images.append(img)
            good_ids.append(image_id)

        if len(images) == 0:
            tries = 0
            while len(images) == 0 and tries < 10:
                image_id = sample(self.image_list, 1)[0]
                img_path = os.path.join(self.path, image_id + ".jpg")
                img = read_image_rgb_128(img_path)
                if img is not None:
                    images.append(img)
                    good_ids.append(image_id)
                tries += 1

        images = np.asarray(images, dtype=np.float32)

        if images.shape[0] == 0:
            images = np.zeros((1, 128, 128, 3), dtype=np.float32)
            if self.labels is None:
                labels = np.zeros((1, 1), dtype=np.float32)
            else:
                mean_label = float(self.labels.mean()) / 100.0
                labels = np.array([[mean_label]], dtype=np.float32)
            return images, labels

        if self.labels is None:
            labels = np.zeros((images.shape[0], 1), dtype=np.float32)
            return images, labels

        labels = self.labels.loc[good_ids].to_numpy().astype(np.float32) / 100.0
        labels = labels.reshape(-1, 1)
        return images, labels

    def loader(self, batch_size=1, shuffle=False):
        while True:
            x, y = self.load_batch(batch_size, shuffle)
            yield x, y


path = os.path.join(BASE, "train")
labels = dat.set_index("Id")["Pawpularity"]
ids = dat["Id"].to_list()

c = data(path, labels=labels, ids=ids)



In [6]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.optimizers import Adam

tf.keras.utils.set_random_seed(42)



2026-01-23 10:36:54.472317: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769164614.485795      12 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769164614.489984      12 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [7]:
model = Sequential()
model.add(Input(shape=(128, 128, 3)))
model.add(Conv2D(10, 5))
model.add(MaxPooling2D())
model.add(Conv2D(20, 3))
model.add(MaxPooling2D())
model.add(Flatten())
model.add(Dense(10, activation="relu"))
model.add(Dense(1, activation="relu"))



I0000 00:00:1769164621.366124      12 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:45:00.0, compute capability: 8.6


In [8]:
model.summary()



Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 124, 124, 10)   │           760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 62, 62, 10)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 60, 60, 20)     │         1,820 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 20)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 18000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │       180,010 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            11 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 182,601 (713.29 KB)

 Trainable params: 182,601 (713.29 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
model.compile(optimizer=Adam(1e-3), loss="mae")



In [10]:
ids = dat["Id"].to_list()
train_ids = ids[: int(len(ids) * 0.8)]
val_ids = ids[int(len(ids) * 0.8) : int(len(ids) * 0.9)]
test_ids = ids[int(len(ids) * 0.9) :]

path = os.path.join(BASE, "train")
labels = dat.set_index("Id")["Pawpularity"]
c = data(path, labels=labels, ids=ids)
c_train = data(path, labels=labels, ids=train_ids)
c_val = data(path, labels=labels, ids=val_ids)
c_test = data(path, labels=labels, ids=test_ids)



In [11]:
model.fit(
    c.loader(batch_size=128, shuffle=True), steps_per_epoch=100, epochs=2, verbose=1
)
model.fit(
    c.loader(batch_size=32, shuffle=True), steps_per_epoch=100, epochs=1, verbose=1
)



Epoch 1/2


I0000 00:00:1769164632.847647      66 service.cc:148] XLA service 0x7f0378003940 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1769164632.847689      66 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
I0000 00:00:1769164632.953773      66 cuda_dnn.cc:529] Loaded cuDNN version 90300


  1/100 ━━━━━━━━━━━━━━━━━━━━ 3:46 2s/step - loss: 0.1929

I0000 00:00:1769164634.173279      66 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


100/100 ━━━━━━━━━━━━━━━━━━━━ 274s 3s/step - loss: 0.1978
Epoch 2/2
100/100 ━━━━━━━━━━━━━━━━━━━━ 84s 844ms/step - loss: 0.1516
100/100 ━━━━━━━━━━━━━━━━━━━━ 15s 148ms/step - loss: 0.1560


In [12]:
testpath = os.path.join(BASE, "test")
test_df = pd.read_csv(os.path.join(BASE, "test.csv"))
ids = test_df["Id"].tolist()



In [13]:
batch_size = 200
y_pred = np.full((len(ids), 1), np.nan, dtype=np.float32)

for start in range(0, len(ids), batch_size):
    end = min(start + batch_size, len(ids))
    batch_ids = ids[start:end]

    images = []
    valid_positions = []
    for j, image_id in enumerate(batch_ids):
        img_path = os.path.join(testpath, image_id + ".jpg")
        img = read_image_rgb_128(img_path)
        if img is None:
            continue
        images.append(img)
        valid_positions.append(j)

    if len(images) == 0:
        continue

    images = np.asarray(images, dtype=np.float32)
    preds = model.predict(images, verbose=0).astype(np.float32) * 100.0
    for k, j in enumerate(valid_positions):
        y_pred[start + j, 0] = preds[k, 0]



In [14]:
fallback = float(dat["Pawpularity"].mean())
y_pred = np.where(np.isfinite(y_pred), y_pred, fallback).astype(np.float32)

alpha = 0.80  # weight on model prediction; (1-alpha) on baseline
y_pred = alpha * y_pred + (1.0 - alpha) * fallback

y_pred = np.clip(y_pred, 1.0, 100.0)

round2 = lambda x, y=None: round(float(x) + 1e-15, y)
y_pred_list = [round2(t[0], 2) for t in y_pred.reshape(-1, 1)]



In [15]:
df = pd.DataFrame()
df["Id"] = ids
df["Pawpularity"] = y_pred_list

assert list(df.columns) == ["Id", "Pawpularity"]
assert len(df) == len(test_df)
assert df["Id"].isna().sum() == 0
assert df["Pawpularity"].isna().sum() == 0



In [16]:
df.to_csv("submission.csv", index=False)
assert os.path.exists("submission.csv") and os.path.getsize("submission.csv") > 0



In [17]:
df.head()

,Id,Pawpularity
0,ee51b99832f1ba868f646df93d2b6b81,29.60
1,caddfb3f8bff9c4b95dbe022018eea21,32.39
2,582eeabd4a448a53ebb79995888a4b0b,34.58
3,afc1ad7f0c5eea880759d09e77f7deee,33.08
4,d5bdf3446e86ce4ec67ce7a00f1cccc2,35.68
